# Databricks: Author and deploy an MCP agent with the Open AI Agents SDK

This notebook shows you how to build an agent using the Open AI Agents SDK that connects to MCP servers hosted on Databricks. The Agents SDK is optimized for Databricks infrastructure and provides automatic tool discovery from MCP servers.

In this notebook, you:

- Author an agent using the Databricks Agents SDK
- Connect the agent to MCP servers to access Databricks-hosted tools
- Test the agent and evaluate its responses using MLflow Evaluation
- Log the agent with MLflow and deploy it to a model serving endpoint

This notebook uses the [`ResponsesAgent`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ResponsesAgent) interface for compatibility with Databricks AI features. In this notebook you learn to:


To learn more about authoring an agent using Agent Framework, see Databricks documentation ([AWS](https://docs.databricks.com/aws/agents/custom-agents/author-agent) | [Azure](https://learn.microsoft.com/azure/databricks/agents/custom-agents/create-chat-model)).

## Prerequisites
- Address all `TODO`s in this notebook.

In [0]:
%pip install --force-reinstall databricks-openai databricks-agents uv
dbutils.library.restartPython()

## Define the agent in code

Define the agent code in a single cell below. This lets you easily write the agent
code to a local Python file, using the `%%writefile` magic command, for subsequent
logging and deployment.

**What this code does at a high level:**

1. **Connect to MCP servers using adapters**
    The `McpServer` adapter from `databricks_openai.agents.mcp_server` handles:
    - Async connections to Databricks MCP servers
    - Authentication (PAT for managed servers, OAuth/M2M for custom servers)
    - Context management (automatically closes connections when done)

2. **Create an agent with MCP tools**
    The Agents SDK `Agent` class automatically discovers tools from connected
    MCP servers and makes them available for the LLM to call.

3. **Handle streaming responses**
    The `MCPToolCallingAgent` class wraps the Databricks Agents SDK agent to:
    - Process streaming events from the OpenAI Responses API
    - Convert events to Databricks-compatible format
    - Enable MLflow tracing for agent execution

4. **Wrap with ResponsesAgent**
    The agent is wrapped using `ResponsesAgent` for compatibility with Databricks
    features like evaluation, deployment, and feedback collection.

5. **MLflow autotracing**
   Enable MLflow autologging to start automatic tracing.

#### Agent tools

This example connects to the Unity Catalog functions MCP server to access
`system.ai.python_exec` (a built-in Python code interpreter). The code also
includes commented-out examples for connecting to:
- Custom MCP servers (hosted as Databricks Apps)
- Vector search MCP servers (with OBO authentication)

In [0]:
%%writefile agent.py
import asyncio
import mlflow
import nest_asyncio

from agents import Agent, set_default_openai_api, set_default_openai_client, Runner
from agents.mcp import MCPServerStreamableHttpParams
from agents.tracing import set_trace_processors
from agents.result import StreamEvent
from databricks_openai.agents.mcp_server import McpServer
from databricks_openai import AsyncDatabricksOpenAI
from databricks.sdk import WorkspaceClient
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from typing import Generator, AsyncIterator, AsyncGenerator
from uuid import uuid4

nest_asyncio.apply()

# Setup mlflow tracing
mlflow.openai.autolog()
# Configure the Databricks OpenAI client and set it to use chat completions in order to access foundation models
set_default_openai_client(AsyncDatabricksOpenAI())
set_default_openai_api("chat_completions")
set_trace_processors([])


############################################
## Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"

# TODO: Update with your system prompt
SYSTEM_PROMPT = """
You are a helpful assistant that can run Python code.
"""

# TODO: Choose your MCP server connection type and setup the Workspace Clients for Authentication

# ---------------------------------------------------------------------------
# Managed MCP Server — simplest setup
# ---------------------------------------------------------------------------
# Databricks manages this connection automatically using your workspace settings
# and Personal Access Token (PAT) authentication.

workspace_client = WorkspaceClient()

host = workspace_client.config.host

# ---------------------------------------------------------------------------
# Custom MCP Server — hosted as a Databricks App
# ---------------------------------------------------------------------------
# Use this if you’re running your own MCP server in Databricks.
# These require OAuth with a service principal for machine-to-machine (M2M) auth.
#
# Follow the insturctions here in order to create a SP, grant the SP query permissions on your app and then mint a client id and # secret. https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m
#
# Uncomment and fill in the settings below to use a custom MCP server.
#
# import os
# custom_mcp_server_workspace_client = WorkspaceClient(
#     host="<DATABRICKS_WORKSPACE_URL>",
#     client_id=os.getenv("DATABRICKS_CLIENT_ID"),
#     client_secret=os.getenv("DATABRICKS_CLIENT_SECRET"),
#     auth_type="oauth-m2m",  # Enables service principal authentication
# )

# ---------------------------------------------------------------------------
# OBO Setup
# ---------------------------------------------------------------------------
# In order to use OBO, uncomment the code below and pass this workspace client to the appropriate McpServer below
#
# from databricks_ai_bridge import ModelServingUserCredentials
# obo_workspace_client = WorkspaceClient(credentials_strategy=ModelServingUserCredentials())


class MCPToolCallingAgent(ResponsesAgent):
    async def process_agent_stream_events(
        self, async_stream: AsyncIterator[StreamEvent]
    ) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
        curr_item_id = str(uuid4())
        async for event in async_stream:
            if event.type == "raw_response_event":
                event_data = event.data.model_dump()
                if (
                    event_data["type"] == "response.output_item.done"
                    and event_data.get("item") is not None
                    and event_data["item"].get("id") is not None
                ):
                    if event_data["item"].get("status") is None:
                        event_data["item"].pop("status", None)
                    event_data["item"]["id"] = curr_item_id
                    event_data.pop("output_index", None)
                    event_data.pop("sequence_number", None)
                    event_data.pop("id", None)
                    yield ResponsesAgentStreamEvent(**event_data)
                elif (
                    event_data["type"] == "response.output_text.delta"
                    and event_data.get("item_id") is not None
                ):
                    event_data["item_id"] = curr_item_id
                    yield ResponsesAgentStreamEvent(**event_data)
            elif (
                event.type == "run_item_stream_event" and event.item.type == "tool_call_output_item"
            ):
                yield ResponsesAgentStreamEvent(
                    type="response.output_item.done",
                    item=event.item.to_input_item(),
                )

    async def _predict_stream_async(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        ###############################################################################
        ## Configure MCP Servers for your agent
        ##
        ## This section sets up server connections so your agent can retrieve data or take actions.

        ## There are three connection types:
        ## 1. Managed MCP servers — fully managed by Databricks
        ## 2. External MCP servers — hosted outside Databricks but proxied through a
        ##    Managed MCP server proxy
        ## 3. Custom MCP servers — MCP servers hosted as Databricks Apps
        ##
        ###############################################################################
        async with (
            McpServer(
                name="system-ai",
                params=MCPServerStreamableHttpParams(
                    url=f"{host}/api/2.0/mcp/functions/system/ai",
                ),
            ) as system_functions,
            # McpServer(
            #     name="custom_app",
            #     params=MCPServerStreamableHttpParams(url="custom_app_url"),
            #     workspace_client=custom_mcp_server_workspace_client,
            # ) as custom_mcp_server,
            # McpServer(
            #     name="vs_obo_mcp",
            #     params=MCPServerStreamableHttpParams(
            #         url=f"{host}/api/2.0/mcp/ai-search/system/ai"
            #     ),
            #     workspace_client=obo_workspace_client,
            # ) as vs_obo_mcp,
        ):
            agent = Agent(
                name="agent",
                instructions=SYSTEM_PROMPT,
                model=LLM_ENDPOINT_NAME,
                mcp_servers=[
                    system_functions,
                ],
            )
            messages = [i.model_dump() for i in request.input]
            result = Runner.run_streamed(agent, input=messages)
            async for event in self.process_agent_stream_events(result.stream_events()):
                yield event

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        agen = self._predict_stream_async(request)

        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)

        ait = agen.__aiter__()

        while True:
            try:
                item = loop.run_until_complete(ait.__anext__())
            except StopAsyncIteration:
                break
            else:
                yield item


mlflow.openai.autolog()
AGENT = MCPToolCallingAgent()
mlflow.models.set_model(AGENT)

## Test the agent

Interact with the agent to test its output. Since we manually traced methods within `ResponsesAgent`, you can view the trace for each step the agent takes, with any LLM calls made via the OpenAI SDK automatically traced by autologging.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
dbutils.library.restartPython()

In [0]:
# ==============================================================================
# TODO: ONLY UNCOMMENT AND EDIT THIS SECTION IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
#       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
# ==============================================================================

# # Set your Databricks client ID and client secret for service principal authentication.
# DATABRICKS_CLIENT_ID = "<YOUR_CLIENT_ID>"
# client_secret_scope_name = "<YOUR_SECRET_SCOPE>"
# client_secret_key_name = "<YOUR_SECRET_KEY_NAME>"

# # Load your service principal credentials into environment variables
# os.environ["DATABRICKS_CLIENT_ID"] = DATABRICKS_CLIENT_ID
# os.environ["DATABRICKS_CLIENT_SECRET"] = dbutils.secrets.get(scope=client_secret_scope_name, key=client_secret_key_name)

In [0]:
from agent import AGENT

result = AGENT.predict({"input": [{"role": "user", "content": "What is 6*7 in Python"}]})
print(result.model_dump(exclude_none=True))

In [0]:
from agent import AGENT

for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "What is 6*7 in Python?"}]}
):
    print(chunk.model_dump(exclude_none=True))

## Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`

  - **TODO**: If your Unity Catalog tool queries a [vector search index](docs link) or leverages [external functions](docs link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/agents/custom-agents/model-serving/agent-authentication-model-serving.html#log-resources-automatic) | [Azure](https://learn.microsoft.com/azure/databricks/agents/custom-agents/model-serving/log-agent#resources)).

In [0]:
import mlflow
from agent import LLM_ENDPOINT_NAME
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction
from pkg_resources import get_distribution

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME), DatabricksFunction(function_name="system.ai.python_exec")]

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            f"databricks-mcp=={get_distribution('databricks-mcp').version}",
            "backoff",
            f"mlflow=={get_distribution('mlflow').version}",
            f"mcp=={get_distribution('mcp').version}",
            f"databricks-openai=={get_distribution('databricks-openai').version}"
        ]
    )

## Evaluate the agent with Agent Evaluation

Use Agent Evaluation to evalaute the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics.
See Databricks documentation ([AWS]((https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).


To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Calculate the 15th Fibonacci number"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
import mlflow

mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Hello!"}]},
    env_manager="uv",
)

## Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.

- **TODO** Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = ""
schema = ""
model_name = ""
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

## Deploy the agent

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME, 
    uc_registered_model_info.version,
    environment_vars={
    # ==============================================================================
    # TODO: ONLY UNCOMMENT AND CONFIGURE THE ENVIRONMENT_VARS SECTION BELOW
    #       IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
    #       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
    # ==============================================================================
    #     "DATABRICKS_CLIENT_ID": DATABRICKS_CLIENT_ID,
    #     "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{client_secret_scope_name}/{client_secret_key_name}}}}}"
    },
    tags = {"endpointSource": "docs"},
    deploy_feedback_model=False
)

## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See docs ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)) for details